In [24]:
import numpy
from hmmlearn import hmm
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

#regimes (trending, mean reverting, high volatility, low volatility)

In [25]:
import yfinance as yf

def clean_yfinance_data(ticker, start, end):
    """Baixa dados e remove MultiIndex se existir"""
    data = yf.download(ticker, start=start, end=end)
    
    # Se tiver MultiIndex nas colunas, remove
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.droplevel(1)
    
    return data['Close']

# Usar a função
sp500 = clean_yfinance_data("^GSPC", "2000-01-01", "2025-12-31")
ibov = clean_yfinance_data("^BVSP", "2000-01-01", "2025-12-31")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [26]:
def prepare_features(prices, window=90):
    """prepara feature para os 4 estados"""
    df = pd.DataFrame(prices)

    df['returns'] = np.log(prices / prices.shift(1))

    df['volatility'] = df['returns'].rolling(window).std()

    df['momentum'] = (prices - prices.rolling(window).mean()) - 1
    
    df['mean_reversion'] = (df['returns'].rolling(window).mean() * df['returns'].shift(1)).rolling(window).mean()

    df['vol_regime'] = df['volatility'] / df['volatility'].rolling(252).mean()

    return df.dropna()
